# SurfaceTarget-CRC — phase 01 — patch 1 5

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.5
#
# Corrected MSI Annotation Retrieval
# MANTIS + MSIsensor
#
# Objectives:
# - Retrieve specific MSI measurements
# - Validate API responses
# - Check patient overlap
# - Identify duplicate/conflicting measurements
# - Preserve data provenance
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

CBIO_API = "https://www.cbioportal.org/api"

STUDY_ID = "coadread_tcga_pan_can_atlas_2018"

TARGET_ATTRIBUTES = [
    "MSI_SCORE_MANTIS",
    "MSI_SENSOR_SCORE"
]

INVENTORY_PATH = (
    METADATA_DIR / "tcga_rnaseq_file_inventory.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.5 — CORRECTED MSI RETRIEVAL")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load GDC RNA-seq inventory
# ------------------------------------------------------------

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str,
    keep_default_na=False
)

gdc_patients = set(
    inventory["patient_id"].dropna()
)

print("\nGDC RNA-seq patients:", len(gdc_patients))

# ------------------------------------------------------------
# 3. Initialize API session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

# ------------------------------------------------------------
# 4. Retrieve clinical attribute definitions
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLINICAL ATTRIBUTE VALIDATION")
print("-" * 65)

url = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-attributes"
)

response = session.get(
    url,
    params={"pageSize": 10000},
    timeout=60
)

response.raise_for_status()

attributes = response.json()

attribute_lookup = {
    a["clinicalAttributeId"]: a
    for a in attributes
    if a.get("clinicalAttributeId")
}

for attribute_id in TARGET_ATTRIBUTES:

    if attribute_id not in attribute_lookup:
        raise RuntimeError(
            f"Required attribute unavailable: {attribute_id}"
        )

    attribute = attribute_lookup[attribute_id]

    print(f"\nAttribute ID: {attribute_id}")

    print(
        "Display Name:",
        attribute.get("displayName")
    )

    print(
        "Patient-level:",
        attribute.get("patientAttribute")
    )

print("\nBoth MSI attributes verified.")

# ------------------------------------------------------------
# 5. Retrieve MSI measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING MSI MEASUREMENTS")
print("-" * 65)

all_records = []

PAGE_SIZE = 1000

for attribute_id in TARGET_ATTRIBUTES:

    print(f"\nRetrieving {attribute_id}")

    attribute_records = []
    page_number = 0

    while True:

        endpoint = (
            f"{CBIO_API}/studies/"
            f"{STUDY_ID}/clinical-data"
        )

        params = {
            "clinicalDataType": "SAMPLE",
            "attributeId": attribute_id,
            "pageSize": PAGE_SIZE,
            "pageNumber": page_number
        }

        response = session.get(
            endpoint,
            params=params,
            timeout=90
        )

        response.raise_for_status()

        records = response.json()

        if not isinstance(records, list):
            raise RuntimeError(
                "Unexpected API response format."
            )

        # Critical validation:
        # Check that every returned measurement
        # belongs to the requested attribute.

        unexpected = [
            record.get("clinicalAttributeId")
            for record in records
            if record.get("clinicalAttributeId")
            != attribute_id
        ]

        if unexpected:
            raise RuntimeError(
                f"API returned unrelated clinical attributes "
                f"for {attribute_id}: {set(unexpected)}"
            )

        attribute_records.extend(records)

        if len(records) < PAGE_SIZE:
            break

        page_number += 1

        if page_number > 100:
            raise RuntimeError(
                "Pagination safety limit exceeded."
            )

    print(
        "Records retrieved:",
        len(attribute_records)
    )

    if not attribute_records:
        raise RuntimeError(
            f"No measurements returned for {attribute_id}"
        )

    all_records.extend(attribute_records)

# ------------------------------------------------------------
# 6. Create corrected MSI dataframe
# ------------------------------------------------------------

msi_df = pd.DataFrame(all_records)

required_columns = [
    "patientId",
    "sampleId",
    "clinicalAttributeId",
    "value"
]

missing_columns = [
    col for col in required_columns
    if col not in msi_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing columns: {missing_columns}"
    )

# Ensure only the requested attributes exist.

returned_attributes = set(
    msi_df["clinicalAttributeId"].unique()
)

assert returned_attributes == set(TARGET_ATTRIBUTES), (
    "Unexpected MSI attributes returned."
)

# Remove exact duplicate records.

msi_df = msi_df.drop_duplicates()

# Convert scores to numeric values.

msi_df["numeric_value"] = pd.to_numeric(
    msi_df["value"],
    errors="coerce"
)

non_numeric = msi_df[
    msi_df["numeric_value"].isna()
]

print("\n" + "-" * 65)
print("DATA VALIDATION")
print("-" * 65)

print(
    "\nTotal corrected MSI records:",
    len(msi_df)
)

print(
    "Non-numeric measurements:",
    len(non_numeric)
)

if not non_numeric.empty:
    print("\nWARNING: Non-numeric measurements found.")
    print(non_numeric.head(10).to_string(index=False))

# ------------------------------------------------------------
# 7. Examine individual MSI measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI MEASUREMENT SUMMARY")
print("-" * 65)

for attribute_id in TARGET_ATTRIBUTES:

    subset = msi_df[
        msi_df["clinicalAttributeId"] == attribute_id
    ]

    scores = subset["numeric_value"].dropna()

    print(f"\n{attribute_id}")

    print("Records:", len(subset))

    print(
        "Unique patients:",
        subset["patientId"].nunique()
    )

    print(
        "Unique samples:",
        subset["sampleId"].nunique()
    )

    print("Numeric scores:", len(scores))

    if not scores.empty:

        print(f"Minimum: {scores.min():.4f}")

        print(f"Median:  {scores.median():.4f}")

        print(f"Maximum: {scores.max():.4f}")

# ------------------------------------------------------------
# 8. Identify duplicate or conflicting measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DUPLICATE / CONFLICT AUDIT")
print("-" * 65)

# Multiple identical values for the same patient
# are different from conflicting values.

patient_measurement_counts = (
    msi_df.groupby(
        ["patientId", "clinicalAttributeId"]
    )["numeric_value"]
    .nunique()
)

conflicting_measurements = (
    patient_measurement_counts[
        patient_measurement_counts > 1
    ]
)

print(
    "\nPatient-attribute combinations with "
    "conflicting numeric scores:",
    len(conflicting_measurements)
)

if not conflicting_measurements.empty:

    print("\nExamples of conflicts:")

    print(
        conflicting_measurements
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 9. Check GDC patient overlap
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GDC / CBIOPORTAL PATIENT MATCHING")
print("-" * 65)

mantis_patients = set(
    msi_df.loc[
        (msi_df["clinicalAttributeId"] == "MSI_SCORE_MANTIS")
        & msi_df["numeric_value"].notna(),
        "patientId"
    ]
)

sensor_patients = set(
    msi_df.loc[
        (msi_df["clinicalAttributeId"] == "MSI_SENSOR_SCORE")
        & msi_df["numeric_value"].notna(),
        "patientId"
    ]
)

both_patients = (
    mantis_patients.intersection(sensor_patients)
)

print("\nGDC RNA-seq patients:", len(gdc_patients))

print(
    "Patients with MANTIS scores:",
    len(mantis_patients)
)

print(
    "Patients with MSIsensor scores:",
    len(sensor_patients)
)

print(
    "Patients with both MSI scores:",
    len(both_patients)
)

print("\nOverlapping with GDC RNA-seq:")

print(
    "MANTIS:",
    len(gdc_patients & mantis_patients)
)

print(
    "MSIsensor:",
    len(gdc_patients & sensor_patients)
)

print(
    "Both measurements:",
    len(gdc_patients & both_patients)
)

# ------------------------------------------------------------
# 10. Project-specific patient overlap
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROJECT-SPECIFIC OVERLAP")
print("-" * 65)

for project_id in ["TCGA-COAD", "TCGA-READ"]:

    project_patients = set(
        inventory.loc[
            inventory["project"] == project_id,
            "patient_id"
        ]
    )

    print(f"\n{project_id}")

    print(
        "RNA-seq patients:",
        len(project_patients)
    )

    print(
        "With MANTIS:",
        len(project_patients & mantis_patients)
    )

    print(
        "With MSIsensor:",
        len(project_patients & sensor_patients)
    )

    print(
        "With both:",
        len(project_patients & both_patients)
    )

# ------------------------------------------------------------
# 11. Save corrected MSI dataset
# ------------------------------------------------------------

output_path = (
    METADATA_DIR /
    "tcga_pancancer_msi_clinical_corrected.csv"
)

msi_df.to_csv(
    output_path,
    index=False
)

# ------------------------------------------------------------
# 12. Save provenance metadata
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "study_id": STUDY_ID,

    "attributes": TARGET_ATTRIBUTES,

    "corrected_msi_records": len(msi_df),

    "gdc_rnaseq_patients": len(gdc_patients),

    "patients_with_mantis": len(mantis_patients),

    "patients_with_msisensor": len(sensor_patients),

    "patients_with_both": len(both_patients),

    "gdc_overlap_both": len(
        gdc_patients & both_patients
    ),

    "conflicting_patient_attribute_measurements": len(
        conflicting_measurements
    ),

    "msi_classification_performed": False,

    "note": (
        "Corrected retrieval using clinicalAttributeId. "
        "No MSS/MSI-H classification applied. "
        "Patient-level score conflicts must be resolved "
        "before cohort assignment."
    )
}

provenance_path = (
    METADATA_DIR /
    "tcga_msi_corrected_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 13. Final status
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.5 SUMMARY")
print("=" * 65)

print("\nCorrected MSI dataset saved:")
print(output_path)

print("\nProvenance metadata saved:")
print(provenance_path)

print("\nNo MSS/MSI-H labels assigned.")
print("No molecular datasets downloaded.")

print("\nPATCH 1.5 COMPLETE")